# RAG for NVDA NDocs

**Pipeline Overview:**
1. Exploratory Data Analysis
2. Data Preparation
3. Build Retrieval Indexes (BM25 + TF-IDF)
4. Feature Engineering
5. Train Ranking Models (Decision Tree, Random Forest, XGBoost, LambdaMART)
6. Evaluation & Comparison
7. End-to-End RAG Demo

## Stage 0: Setup & Install Dependencies

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)

## Stage 1: EDA

In [ ]:
from datasets import load_dataset

#load dataset
dataset = load_dataset("nvidia/Retrieval-Synthetic-NVDocs-v1")

print(dataset)

In [ ]:
display(dataset['train'][0])

In [ ]:
from datasets import DatasetDict

full_dataset = dataset['train']

train_testval = full_dataset.train_test_split(test_size=0.2, seed=42)

In [ ]:
#split test_val into validation (10%) and test (10%)
test_val = train_testval['test'].train_test_split(test_size=0.5, seed=42)

In [ ]:
#build DatasetDict
split_dataset = DatasetDict({
    'train': train_testval['train'],
    'validation': test_val['train'],
    'test': test_val['test']
})

In [ ]:
print(split_dataset)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

#train split to DataFrame
df = split_dataset['train'].to_pandas()

In [ ]:
df.info()

In [ ]:
#text length distribution
df['text_length'] = df['text'].apply(lambda x: len(str(x)))

plt.figure(figsize=(10, 5))
plt.hist(df['text_length'], bins=50, color='skyblue', edgecolor='black')
plt.title('Distribution of Text Lengths')
plt.xlabel('Length of Text (characters)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
#is_multi_doc distribution
if 'is_multi_doc' in df.columns:
    print(df['is_multi_doc'].value_counts())

### Deep dive: Understand the QA-Chunk mapping

Each document has chunks and QA pairs. Each QA pair's `segment_ids` tells us which chunks are relevant.
This is our **golden label**

In [ ]:
#single document structure
record = split_dataset['train'][0]

print(f"File: {record['file_name']}")
print(f"Number of chunks: {len(record['chunks'])}")
print(f"Number of QA pairs: {len(record['deduplicated_qa_pairs'])}")

In [ ]:
#first QA pair details
qa = record['deduplicated_qa_pairs'][0]
print(f"Question: {qa['question']}")
print(f"Answer: {qa['answer'][:200]}...")
print(f"Query Type: {qa['query_type']}")
print(f"Reasoning Type: {qa['reasoning_type']}")
print(f"Complexity: {qa['question_complexity']}")
print(f"Hop Count: {qa['hop_count']}")
print(f"Relevant Segment IDs: {qa['segment_ids']}")

In [ ]:
#relevant chunks preview
chunk_lookup = {c['chunk_id']: c for c in record['chunks']}
for sid in qa['segment_ids']:
    if sid in chunk_lookup:
        c = chunk_lookup[sid]
        print(f"\nChunk ID: {sid}")
        print(f"Text preview: {c['text'][:300]}...")
        print(f"Word count: {c['word_count']}, Sentence count: {c['sentence_count']}")

## Stage 2: Data Preparation

We extract all chunks and QA pairs, then create training triples:
- **Positive**: (question, relevant_chunk, 1) — chunk is in `segment_ids`
- **Negative**: (question, random_chunk, 0) — sampled from other chunks

After running the first round of training and validating. We learned that using random chunks led to subpar NDCG@10 (same as BM25 raw ranking). We improved the random chunks to be instead candidates from the BM25-TFIDF top 100 candidate pools for improving accuracy. 

In [ ]:
from data_prep import extract_all_chunks, extract_qa_pairs, create_relevance_triples

#extract train chunks and QA pairs
train_chunks = extract_all_chunks(split_dataset['train'], doc_offset=0)
train_qa = extract_qa_pairs(split_dataset['train'], doc_offset=0)
print(f"Train: {len(train_chunks)} chunks, {len(train_qa)} QA pairs")

In [ ]:
#extract validation chunks and QA pairs
val_offset = len(split_dataset['train'])
val_chunks = extract_all_chunks(split_dataset['validation'], doc_offset=val_offset)
val_qa = extract_qa_pairs(split_dataset['validation'], doc_offset=val_offset)
print(f"Val: {len(val_chunks)} chunks, {len(val_qa)} QA pairs")

In [ ]:
#extract test chunks and QA pairs
test_offset = val_offset + len(split_dataset['validation'])
test_chunks = extract_all_chunks(split_dataset['test'], doc_offset=test_offset)
test_qa = extract_qa_pairs(split_dataset['test'], doc_offset=test_offset)
print(f"Test: {len(test_chunks)} chunks, {len(test_qa)} QA pairs")

In [ ]:
#global chunk pool for retrieval
all_chunks = train_chunks + val_chunks + test_chunks
print(f"Total chunks in corpus: {len(all_chunks)}")

In [ ]:
#create train triples
train_triples = create_relevance_triples(train_qa, all_chunks, neg_per_positive=4, seed=42)
print(f"Train triples: {len(train_triples)} ({train_triples['relevance'].sum()} pos, {(train_triples['relevance']==0).sum()} neg)")

In [ ]:
#create validation triples
val_triples = create_relevance_triples(val_qa, all_chunks, neg_per_positive=4, seed=42)
print(f"Val triples: {len(val_triples)} ({val_triples['relevance'].sum()} pos, {(val_triples['relevance']==0).sum()} neg)")

Check the Chunk IDs, Items

In [ ]:
chunk_lookup_all = {c['chunk_id']: c for c in all_chunks}

#No duplicate chunk IDs across splits
all_ids = [c['chunk_id'] for c in all_chunks]
assert len(all_ids) == len(set(all_ids)), f"DUPLICATE chunk IDs! {len(all_ids)} total vs {len(set(all_ids))} unique"
print(f"✓ All {len(all_ids)} chunk IDs are globally unique")

In [ ]:
#Golden segment_id coverage
for split_name, qa_list in [('train', train_qa), ('val', val_qa), ('test', test_qa)]:
    missing = 0
    total = 0
    for qa in qa_list:
        for sid in qa['segment_ids']:
            total += 1
            if sid not in chunk_lookup_all:
                missing += 1
    pct = missing / total * 100 if total > 0 else 0
    if missing == 0:
        print(f"✓ {split_name}: all {total} golden segment_ids resolve to real chunks")
    else:
        print(f"⚠ {split_name}: {missing}/{total} ({pct:.2f}%) golden segment_ids not in corpus (dataset noise, skipped in training)")

In [ ]:
#Positive triples actually match their golden chunks
for split_name, triples in [('train', train_triples), ('val', val_triples)]:
    pos_triples = triples[triples['relevance'] == 1]
    bad = 0
    for _, row in pos_triples.iterrows():
        if row['chunk_id'] not in chunk_lookup_all:
            bad += 1
            continue
        actual_text = chunk_lookup_all[row['chunk_id']]['chunk_text']
        if actual_text != row['chunk_text']:
            bad += 1
    assert bad == 0, f"{split_name}: {bad} positive triples have mismatched chunk text!"
    print(f"✓ {split_name}: all {len(pos_triples)} positive triples have correct chunk text")

In [ ]:
#No train/val/test doc_idx overlap
train_docs = set(c['doc_idx'] for c in train_chunks)
val_docs = set(c['doc_idx'] for c in val_chunks)
test_docs = set(c['doc_idx'] for c in test_chunks)
assert not (train_docs & val_docs), f"Train/Val doc overlap: {train_docs & val_docs}"
assert not (train_docs & test_docs), f"Train/Test doc overlap: {train_docs & test_docs}"
assert not (val_docs & test_docs), f"Val/Test doc overlap: {val_docs & test_docs}"
print(f"✓ No doc_idx overlap: train={len(train_docs)}, val={len(val_docs)}, test={len(test_docs)}")

In [ ]:
#Show a sample positive triple with context
sample_pos = train_triples[train_triples['relevance'] == 1].iloc[0]
print(f"\n── Sample positive triple ──")
print(f"  Question: {sample_pos['question'][:120]}...")
print(f"  Chunk ID: {sample_pos['chunk_id']}")
print(f"  Chunk text: {sample_pos['chunk_text'][:120]}...")
print(f"  Relevance: {sample_pos['relevance']}")

In [ ]:
#Label balance
for split_name, triples in [('train', train_triples), ('val', val_triples)]:
    n_pos = (triples['relevance'] == 1).sum()
    n_neg = (triples['relevance'] == 0).sum()
    ratio = n_neg / n_pos if n_pos > 0 else float('inf')
    print(f"\n  {split_name}: {n_pos} positive, {n_neg} negative (ratio ~{ratio:.1f}:1)")

There are 332 (out of ~192k chunks) in the train split that don't have a matching chunk in the corpus. We will need to account for this in the training pipeline when a chunk is missing. 

## Stage 3: Build Retrieval Indexes

We build two indexes over the **entire corpus** of chunks:

1. **BM25** (sparse, lexical): Scores based on token overlap
2. **TF-IDF** (sparse, semantic): Uses sklearn's TfidfVectorizer + cosine similarity. Captures term importance.

These scores become **features** for our RAG Models.

In [ ]:
from retrieval import BM25Index, TfidfIndex

#prepare corpus texts and IDs
corpus_texts = [c['chunk_text'] for c in all_chunks]
corpus_ids = [c['chunk_id'] for c in all_chunks]
print(f"Corpus size: {len(corpus_texts)} chunks")

In [ ]:
#build BM25 index
bm25_index = BM25Index(corpus_texts, corpus_ids)

In [ ]:
#build TF-IDF index
tfidf_index = TfidfIndex(max_features=50000, ngram_range=(1, 2))
tfidf_index.build_index(corpus_texts, corpus_ids)

In [ ]:
#test retrieval top-5
sample_q = train_qa[0]['question']
print(f"Query: {sample_q}\n")

for chunk_id, score in bm25_index.top_k(sample_q, k=5):
    print(f"  {chunk_id}: score={score:.4f}")

for chunk_id, score in tfidf_index.top_k(sample_q, k=5):
    print(f"  {chunk_id}: score={score:.4f}")

#check golden chunk in results
golden_ids = set(train_qa[0]['segment_ids'])
print(f"\nGolden chunk IDs: {golden_ids}")

## Stage 4: Feature Engineering

For each (question, chunk) pair, we compute a feature vector:

| Feature | Type | Source |
|---------|------|--------|
| `bm25_score` | Numeric | BM25 index |
| `cosine_similarity` | Numeric | TF-IDF cosine similarity |
| `token_overlap_ratio` | Numeric | Text overlap |
| `token_overlap_count` | Numeric | Text overlap |
| `question_length` | Numeric | Question metadata |
| `chunk_word_count` | Numeric | Chunk metadata |
| `chunk_sentence_count` | Numeric | Chunk metadata |
| `question_complexity` | Numeric | QA metadata |
| `hop_count` | Numeric | QA metadata |
| `qtype_*` | One-hot | Query type |
| `rtype_*` | One-hot | Reasoning type |

The ranking models will learn which combinations of these features predict relevance.

In [ ]:
from features import compute_features, get_feature_columns

#compute train features
train_featured = compute_features(train_triples, bm25_index, tfidf_index)
print(f"\nTrain shape: {train_featured.shape}")

In [ ]:
#inspect features
feature_cols = get_feature_columns(train_featured)
print(f"Feature columns ({len(feature_cols)}):")
for col in feature_cols:
    print(f"  - {col}")

train_featured[feature_cols].describe().round(3)

In [ ]:
import seaborn as sns

#feature distributions by relevance
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
key_features = ['bm25_score', 'cosine_similarity', 'token_overlap_ratio',
                'question_length', 'chunk_word_count', 'question_complexity']

for ax, feat in zip(axes.flat, key_features):
    for rel, color, label in [(1, 'green', 'Relevant'), (0, 'red', 'Not Relevant')]:
        subset = train_featured[train_featured['relevance'] == rel][feat]
        ax.hist(subset, bins=30, alpha=0.5, color=color, label=label, density=True)
    ax.set_title(feat)
    ax.legend()

plt.suptitle('Feature Distributions by Relevance', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier as _RFC

#quick RF for feature importance analysis
_rf_importance = _RFC(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
_rf_importance.fit(train_featured[feature_cols].values, train_featured['relevance'].values)

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': _rf_importance.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importance (Random Forest):\n")
print(importance_df.to_string(index=False))

In [ ]:
#visualize
fig, ax = plt.subplots(figsize=(10, 6))
importance_df.plot(x='feature', y='importance', kind='barh', ax=ax, legend=False)
ax.invert_yaxis()
ax.set_title('Feature Importance (Random Forest)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
#identify low-importance features to drop
drop_threshold = 0.01
low_importance = importance_df[importance_df['importance'] < drop_threshold]['feature'].tolist()
print(f"\nFeatures below {drop_threshold} threshold (candidates to drop):")
for f in low_importance:
    print(f"  - {f}")

## Stage 5: Train Ranking Models

**Key insight**: Models must train on the **same distribution** they see at evaluation.

At eval time, models re-rank candidates from BM25 ∪ TF-IDF top-100, so **all candidates already have high retrieval scores**. If we train on random negatives (low BM25/TF-IDF), models learn a coarse "high BM25 = relevant" rule that can't discriminate within the candidate pool.

**Solution**: Build training data from candidate pools (hard negatives), exactly matching the eval setup.

**Models:**
1. **BM25 Baseline** — no training, use BM25 scores directly
2. **TF-IDF Baseline** — no training, use TF-IDF cosine similarity
3. **Decision Tree** — simple tree-based classifier
4. **Random Forest** — ensemble of trees
5. **XGBoost Ranker** — gradient-boosted trees with `rank:ndcg` objective
6. **LightGBM Ranker** — LambdaMART implementation

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from evaluate import prepare_candidates, evaluate_from_candidates, candidates_to_training_data, print_evaluation

In [ ]:
#drop low-importance qtype/rtype one-hot features
drop_features = [c for c in feature_cols if c.startswith('qtype_') or c.startswith('rtype_')]
feature_cols = [c for c in feature_cols if c not in drop_features]
print(f"Dropped {len(drop_features)} low-importance features: {drop_features}")
print(f"Keeping {len(feature_cols)} features: {feature_cols}")

In [ ]:
#chunk lookup for full-corpus candidate retrieval
chunk_lookup = {c['chunk_id']: c for c in all_chunks}

#precompute TRAIN candidates (same BM25 ∪ TF-IDF top-100 pool as eval)
print("Precomputing train candidates (BM25 ∪ TF-IDF top-100)...")
train_candidates = prepare_candidates(train_qa, bm25_index, tfidf_index,
                                       chunk_lookup, feature_cols, candidate_k=100)
print(f"Train: {len(train_candidates)} questions, avg {np.mean([c['metadata']['n_candidates'] for c in train_candidates]):.0f} candidates each")

#precompute VAL candidates
print("\nPrecomputing val candidates (BM25 ∪ TF-IDF top-100)...")
val_candidates = prepare_candidates(val_qa, bm25_index, tfidf_index,
                                     chunk_lookup, feature_cols, candidate_k=100)
print(f"Val: {len(val_candidates)} questions, avg {np.mean([c['metadata']['n_candidates'] for c in val_candidates]):.0f} candidates each")

### 5a. Build Training Data from Candidates & Baselines

Training data uses the **same candidate pool** as evaluation — this eliminates the
distribution shift between random negatives (low BM25) and eval candidates (high BM25).

In [ ]:
#convert candidates to training arrays
X_train, y_train, train_groups = candidates_to_training_data(train_candidates, feature_cols)
X_val, y_val, val_groups = candidates_to_training_data(val_candidates, feature_cols)

print(f"Training: X={X_train.shape}, positives={y_train.sum():.0f}, negatives={(y_train==0).sum():.0f}")
print(f"Validation: X={X_val.shape}, positives={y_val.sum():.0f}, negatives={(y_val==0).sum():.0f}")
print(f"Train groups: {len(train_groups)} questions, avg group size: {train_groups.mean():.0f}")
print(f"Val groups: {len(val_groups)} questions, avg group size: {val_groups.mean():.0f}")

#verify distribution matches eval
print(f"\nTrain feature means: {X_train.mean(axis=0).round(4)}")
print(f"Val feature means:   {X_val.mean(axis=0).round(4)}")
print(f"Feature names: {feature_cols}")

In [ ]:
#BM25 baseline on validation
bm25_val_metrics, _ = evaluate_from_candidates(val_candidates, model=None)
print_evaluation(bm25_val_metrics, "BM25 Baseline (Val)")

#TF-IDF baseline on validation
tfidf_val_metrics, _ = evaluate_from_candidates(val_candidates, model='tfidf')
print_evaluation(tfidf_val_metrics, "TF-IDF Cosine Similarity Baseline (Val)")

### 5b. Decision Tree Classifier

In [ ]:
#train decision tree
dt_model = DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, random_state=42)
dt_model.fit(X_train, y_train)

In [ ]:
#evaluate decision tree on validation 
dt_val_metrics, _ = evaluate_from_candidates(val_candidates, dt_model)
print_evaluation(dt_val_metrics, "Decision Tree (Val)")

dt_importance = pd.Series(dt_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(dt_importance.head(10))

### 5c. Random Forest Classifier

In [ ]:
#train random forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

In [ ]:
#evaluate random forest on validation
rf_val_metrics, _ = evaluate_from_candidates(val_candidates, rf_model)
print_evaluation(rf_val_metrics, "Random Forest (Val)")

rf_importance = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(rf_importance.head(10))

### 5d. XGBoost Ranker (LambdaMART)

XGBoost's `rank:ndcg` objective directly optimizes NDCG — the key ranking metric.
It needs a `group` parameter telling it which rows belong to the same query.

In [ ]:
import xgboost as xgb

#groups already computed by candidates_to_training_data
print(f"Train groups: {len(train_groups)}, Val groups: {len(val_groups)}")
print(f"X_train: {X_train.shape}, X_val: {X_val.shape}")

In [ ]:
#train XGBoost ranker
xgb_model = xgb.XGBRanker(
    objective='rank:ndcg',
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method='hist',
)

xgb_model.fit(
    X_train, y_train,
    group=train_groups,
    eval_set=[(X_val, y_val)],
    eval_group=[val_groups],
    verbose=50
)

In [ ]:
#evaluate XGBoost on validation (full-corpus)
xgb_val_metrics, _ = evaluate_from_candidates(val_candidates, xgb_model)
print_evaluation(xgb_val_metrics, "XGBoost Ranker (Val)")

### 5e. LightGBM Ranker (LambdaMART)

In [ ]:
import lightgbm as lgb

#train LightGBM ranker
lgb_model = lgb.LGBMRanker(
    objective='lambdarank',
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
)

lgb_model.fit(
    X_train, y_train,
    group=train_groups,
    eval_set=[(X_val, y_val)],
    eval_group=[val_groups],
    eval_at=[5, 10, 20],
)

In [ ]:
#evaluate LightGBM on validation (full-corpus)
lgb_val_metrics, _ = evaluate_from_candidates(val_candidates, lgb_model)
print_evaluation(lgb_val_metrics, "LightGBM Ranker (Val)")

## Stage 6: Model Comparison & Final Test Evaluation

Compare all models side-by-side on validation, pick the best, then do final evaluation on the held-out test set.

In [ ]:
#validation metrics comparison
all_val_metrics = {
    'BM25 Baseline': bm25_val_metrics,
    'TF-IDF Cosine': tfidf_val_metrics,
    'Decision Tree': dt_val_metrics,
    'Random Forest': rf_val_metrics,
    'XGBoost Ranker': xgb_val_metrics,
    'LightGBM Ranker': lgb_val_metrics,
}

comparison_df = pd.DataFrame(all_val_metrics).T
comparison_df = comparison_df.round(4)
display(comparison_df)

In [ ]:
#visualize model comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics_to_plot = ['NDCG@10', 'Recall@10', 'MRR']
for ax, metric in zip(axes, metrics_to_plot):
    values = comparison_df[metric].sort_values(ascending=True)
    colors = ['#2ecc71' if v == values.max() else '#3498db' for v in values]
    values.plot(kind='barh', ax=ax, color=colors)
    ax.set_title(metric, fontsize=14)
    ax.set_xlabel('Score')
    for i, v in enumerate(values):
        ax.text(v + 0.005, i, f'{v:.4f}', va='center')

plt.suptitle('Model Comparison on Validation Set', fontsize=16)
plt.tight_layout()
plt.show()

### Final Test Evaluation (Full-Corpus Retrieval)

For each test question, we retrieve **top-100 candidates from BM25 ∪ top-100 from TF-IDF**, then re-rank with the best model. Metrics are computed over this realistic candidate pool (~150-200 chunks per question)

In [ ]:
#precompute test candidates + features ONCE
test_candidates = prepare_candidates(test_qa, bm25_index, tfidf_index,
                                      chunk_lookup, feature_cols, candidate_k=100)
print(f"Done. {len(test_candidates)} questions, avg {np.mean([c['metadata']['n_candidates'] for c in test_candidates]):.0f} candidates each")

In [ ]:
#best model by val NDCG@10
best_model_name = comparison_df['NDCG@10'].idxmax()
print(f"Best model (by Val NDCG@10): {best_model_name}")

In [ ]:
#select best model (include baselines so any winner works)
model_objects = {
    'BM25 Baseline': None,
    'TF-IDF Cosine': 'tfidf',
    'Decision Tree': dt_model,
    'Random Forest': rf_model,
    'XGBoost Ranker': xgb_model,
    'LightGBM Ranker': lgb_model,
}
best_model_obj = model_objects[best_model_name]

In [ ]:
#evaluate BM25 baseline 
bm25_test_metrics, bm25_meta = evaluate_from_candidates(test_candidates, model=None)
print_evaluation(bm25_test_metrics, "BM25 Baseline (Test)")

In [ ]:
#evaluate TF-IDF baseline 
tfidf_test_metrics, tfidf_meta = evaluate_from_candidates(test_candidates, model='tfidf')
print_evaluation(tfidf_test_metrics, "TF-IDF Baseline (Test)")

In [ ]:
#evaluate best model 
print(f"Evaluating {best_model_name}...")
best_test_metrics, best_meta = evaluate_from_candidates(test_candidates, best_model_obj)
print_evaluation(best_test_metrics, f"{best_model_name} (Test)")

In [ ]:
#side-by-side comparison
test_comparison_df = pd.DataFrame({
    'BM25 Baseline': bm25_test_metrics,
    'TF-IDF Cosine': tfidf_test_metrics,
    best_model_name: best_test_metrics,
}).T.round(4)

display(test_comparison_df)

In [ ]:
#candidate pool diagnostics
print(f"Avg candidates per question: {best_meta['n_candidates'].mean():.0f}")
print(f"Avg golden chunks per question: {best_meta['n_golden'].mean():.1f}")
print(f"Golden chunk recall in candidate pool: {(best_meta['golden_in_candidates'] / best_meta['n_golden']).mean():.4f}")

In [ ]:
#breakdown by query type 
print(f"=== {best_model_name} — Breakdown by Query Type ===\n")
for qtype in best_meta['query_type'].unique():
    mask = best_meta['query_type'] == qtype
    subset = [c for c, m in zip(test_candidates, mask) if m]
    if len(subset) > 0:
        metrics, _ = evaluate_from_candidates(subset, best_model_obj, k_values=[10])
        print(f"  {qtype:20s} — NDCG@10: {metrics['NDCG@10']:.4f}, Recall@10: {metrics['Recall@10']:.4f}, n={len(subset)}")

In [ ]:
#breakdown by reasoning type 
print(f"=== {best_model_name} — Breakdown by Reasoning Type ===\n")
for rtype in best_meta['reasoning_type'].unique():
    mask = best_meta['reasoning_type'] == rtype
    subset = [c for c, m in zip(test_candidates, mask) if m]
    if len(subset) > 0:
        metrics, _ = evaluate_from_candidates(subset, best_model_obj, k_values=[10])
        print(f"  {rtype:20s} — NDCG@10: {metrics['NDCG@10']:.4f}, Recall@10: {metrics['Recall@10']:.4f}, n={len(subset)}")

## Stage 7: End-to-End RAG Retrieval

Given a new question:
1. **First-stage retrieval**: Use TF-IDF to get top-50 candidate chunks (fast, broad recall)
2. **Feature computation**: Compute features for each (question, candidate) pair
3. **Re-rank**: Use the best trained model to re-score and re-rank
4. **Return top-K**: Final ranked list of most relevant chunks 5

In [ ]:
from features import compute_text_overlap

def rag_retrieve(question, tfidf_index, bm25_index, model, feature_cols,
                 candidate_k=50, final_k=5):
    #first-stage: BM25 ∪ TF-IDF retrieval
    bm25_candidates = dict(bm25_index.top_k(question, k=candidate_k))
    tfidf_candidates = dict(tfidf_index.top_k(question, k=candidate_k))

    #merge candidate sets
    seen = set()
    merged = []
    for cid in list(bm25_candidates.keys()) + list(tfidf_candidates.keys()):
        if cid not in seen:
            seen.add(cid)
            merged.append(cid)

    #handle baseline models (no trained model)
    if model is None:
        #BM25 baseline: rank by BM25 score
        results = [(cid, chunk_lookup[cid]['chunk_text'], bm25_candidates.get(cid, 0.0)) for cid in merged]
        results.sort(key=lambda x: x[2], reverse=True)
        return results[:final_k]
    elif model == 'tfidf':
        #TF-IDF baseline: rank by TF-IDF score
        results = [(cid, chunk_lookup[cid]['chunk_text'], tfidf_candidates.get(cid, 0.0)) for cid in merged]
        results.sort(key=lambda x: x[2], reverse=True)
        return results[:final_k]

    #build feature rows per candidate
    chunk_lookup_local = {c['chunk_id']: c for c in all_chunks}
    rows = []
    for chunk_id in merged:
        chunk = chunk_lookup_local[chunk_id]
        overlap = compute_text_overlap(question, chunk['chunk_text'])

        row = {
            'bm25_score': bm25_candidates.get(chunk_id, 0.0),
            'cosine_similarity': tfidf_candidates.get(chunk_id, 0.0),
            'token_overlap_ratio': overlap['token_overlap_ratio'],
            'token_overlap_count': overlap['token_overlap_count'],
            'question_length': len(question.split()),
            'chunk_word_count': chunk['word_count'],
            'chunk_sentence_count': chunk['sentence_count'],
            'question_complexity': 3,
            'hop_count': 1,
        }
        rows.append((chunk_id, chunk['chunk_text'], row))

    #create feature matrix
    feature_rows = pd.DataFrame([r[2] for r in rows])
    for col in feature_cols:
        if col not in feature_rows.columns:
            feature_rows[col] = 0
    feature_rows = feature_rows[feature_cols]

    #re-rank with trained model
    if hasattr(model, 'predict_proba'):
        scores = model.predict_proba(feature_rows.values)[:, 1]
    else:
        scores = model.predict(feature_rows.values)

    #sort by score, return top-K
    results = [(rows[i][0], rows[i][1], float(scores[i])) for i in range(len(rows))]
    results.sort(key=lambda x: x[2], reverse=True)
    return results[:final_k]

In [ ]:
#demo retrieval from test set
demo_question = test_qa[0]['question']
demo_golden = set(test_qa[0]['segment_ids'])

print(f"Query: {demo_question}\n")
print(f"Golden relevant chunks: {demo_golden}\n")

results = rag_retrieve(demo_question, tfidf_index, bm25_index, best_model_obj, feature_cols)

print(f"=== Top-5 Retrieved Chunks ({best_model_name}) ===\n")
for rank, (chunk_id, text, score) in enumerate(results, 1):
    is_relevant = "RELEVANT" if chunk_id in demo_golden else ""
    print(f"Rank {rank}: {chunk_id} (score={score:.4f}) {is_relevant}")
    print(f"  {text[:150]}...")